# 05 · Stage 0: CDI category tables for a comparison dataset

**What it does.** Builds CDI category embeddings for the comparison dataset and rebuilds BV and THINGS on the same
categories: the ImageNet/LVIS analogue of manuscript notebook 07.

**Decisions**
- **Two category sets.** `<dataset>_hyponym<n>`: every valid129 word with a non-empty pool (pools can overlap).
  `<dataset>_exclusive<n>` (**primary**): each class counts only for its closest valid129 word (fewest hypernym
  steps); a tie goes to the larger pool. ImageNet: 79 / 75 categories; LVIS: 105 / 104.
- **Category vector** = mean over all pooled exemplars of the L2-normalized embedding.
- **Re-z-scoring.** The manuscript z-scores each dimension across categories, so a 129-category table is not
  comparable with a 75-category one. BV and THINGS are rebuilt from their **raw** valid129 tables and z-scored within
  each shared set.

**Inputs:** maps (01), class means (03), manuscript raw valid129 tables.
**Outputs:** `outputs/exemplar_sets/<set>/{bv,things,<dataset>}_<model>_exemplar_avg_{raw,zscore}_within_<set>.csv`,
`categories.txt`, `<dataset>_category_classes.csv`, `<dataset>_exemplar_embedding_run.json`.

In [1]:
# Standard setup shared by every notebook (each notebook is self-contained).
import json
import os
import random
import re
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch, Rectangle
from scipy.cluster.hierarchy import leaves_list, linkage, optimal_leaf_ordering
from scipy.spatial.distance import pdist, squareform
from scipy.stats import false_discovery_control, pearsonr, rankdata, spearmanr, wilcoxon
from IPython.display import Image, Markdown, display

plt.rcParams["font.family"] = ["DejaVu Sans", "sans-serif"]
pd.set_option("display.max_colwidth", 120)


def show_images(paths, width=950):
    """Display saved figures (the files this notebook writes)."""
    for p in map(Path, paths):
        if p.exists():
            display(Markdown(f"`{p.relative_to(ROOT)}`")); display(Image(filename=str(p), width=width))
        else:
            display(Markdown(f"*Missing figure:* `{p}`"))

In [2]:
# Paths. Run this notebook from analysis/comparison-datasets/notebooks/ (Jupyter's default working directory).
ROOT = Path.cwd().resolve().parent                     # analysis/comparison-datasets
assert (ROOT / "notebooks").is_dir(), "open this notebook from analysis/comparison-datasets/notebooks/"
REPO = ROOT.parents[1]
DATA_DIR = REPO / "data"
MANUSCRIPT_DIR = REPO / "analysis" / "manuscript-2026"
VEDB_DIR = REPO / "analysis" / "vedb-1612"
VOCAB_DIR = DATA_DIR / "vocabulary_lists"
VALID129_TXT = VOCAB_DIR / "included_categories_valid129.txt"

# Inputs written by other pipelines (read only).
EXEMPLAR_EMBED_ROOT = Path(os.environ.get("BV_EXEMPLAR_EMBED_ROOT",
                                          str(MANUSCRIPT_DIR / "exemplar_set_embeddings"))).expanduser()
BV_THINGS_VALID129 = EXEMPLAR_EMBED_ROOT / "valid129"       # manuscript Stage 0: BV + THINGS tables
VEDB_VALID129 = VEDB_DIR / "exemplar_set_embeddings" / "valid129"   # vedb-1612 Stage 0 (files use the "bv_" prefix)

# External data on ccn2.
IMAGENET_DIR = Path("/data2/j7yang/imagenet-1k/extracted")
IMAGENET_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/imagenet_1k/outputs/image_embeddings")
LVIS_ROOT = Path("/data2/j7yang/lvis")
LVIS_CROP_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/lvis/outputs/crop_embeddings")

# Outputs of this folder.
OUT = ROOT / "outputs"
MAPPING_OUT = OUT / "mapping"
MONTAGE_OUT = OUT / "montages"
EXEMPLAR_SETS = OUT / "exemplar_sets"
MODELS = ("clip", "dinov3")

## Parameters

In [3]:
DATASET = "lvis"
assert DATASET in ('imagenet', 'lvis'), "DATASET must be one of ('imagenet', 'lvis')"

## Specs and helpers

In [4]:
SPECS = {
    "imagenet": {
        "map_csv": "imagenet_cdi_hyponym_map.csv", "key": "wnid", "label": "imagenet_synset",
        "npz": "imagenet_class_embeddings_{}.npz", "npz_ids": "wnids",
        "description": "imagenet-1k (train+val)",
        "anchors": "analysis/comparison-datasets/mapping/cdi_imagenet_anchors.py",
        "exemplar": "per-image L2 norm, then mean over all pooled images",
    },
    "lvis": {
        "map_csv": "lvis_cdi_hyponym_map.csv", "key": "lvis_category_id", "label": "lvis_name",
        "npz": "lvis_category_embeddings_{}.npz", "npz_ids": "category_ids",
        "description": "LVIS v1 ground-truth box crops (train+val annotations; no CLIP or size filter)",
        "anchors": "analysis/comparison-datasets/mapping/cdi_lvis_anchors.py",
        "exemplar": "per-crop L2 norm, then mean over all pooled box crops",
    },
}


def zscore_rows(X: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    """Same as notebook 07: z-score each dimension across categories."""
    return (X - X.mean(axis=0)) / (X.std(axis=0) + eps)


def load_raw_table(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["category"] = df["category"].astype(str).str.strip().str.lower()
    return df.set_index("category")


def save_table(path: Path, cats: list[str], X: np.ndarray) -> None:
    df = pd.DataFrame(X, columns=[f"dim_{i}" for i in range(X.shape[1])])
    df.insert(0, "category", cats)
    df.to_csv(path, index=False)


def category_pools(spec: dict, variant: str) -> pd.DataFrame:
    """(cdi_category, source class) rows for valid129 categories under the given pooling variant."""
    valid129 = set(VALID129_TXT.read_text().split())
    m = pd.read_csv(MAPPING_OUT / spec["map_csv"])
    m = m[m["cdi_category"].isin(valid129)]
    if variant == "exclusive":
        closest = m.groupby(spec["key"])["hypernym_distance"].transform("min")
        m = m[m["hypernym_distance"] == closest]
        ties = m[m[spec["key"]].duplicated(keep=False)]
        if len(ties):
            pool_size = pd.read_csv(MAPPING_OUT / spec["map_csv"]).groupby("cdi_category").size()
            m = (m.assign(_pool=m["cdi_category"].map(pool_size))
                  .sort_values(["_pool", "cdi_category"], ascending=[False, True])
                  .drop_duplicates(spec["key"]).drop(columns="_pool"))
            print(f"  exclusive ties -> larger pool: "
                  f"{ties.groupby(spec['label'])['cdi_category'].apply(list).to_dict()}")
    return m[["cdi_category", spec["key"], spec["label"], "hypernym_distance"]]

## Build the category sets

In [5]:
def build_category_sets(dataset: str):
    """Stage 0 for one comparison dataset ('imagenet' | 'lvis'): write both category sets; return their names."""
    ds_name, spec = dataset, SPECS[dataset]

    src_dir = BV_THINGS_VALID129
    class_emb = {m: np.load(MAPPING_OUT / spec["npz"].format(m)) for m in MODELS}

    set_names = []
    for variant in ("hyponym", "exclusive"):
        pools = category_pools(spec, variant)
        cats = sorted(pools["cdi_category"].unique())
        set_name = f"{ds_name}_{variant}{len(cats)}"
        set_names.append(set_name)
        out_dir = EXEMPLAR_SETS / set_name
        out_dir.mkdir(parents=True, exist_ok=True)
        (out_dir / "categories.txt").write_text("\n".join(cats) + "\n")
        pools.sort_values(["cdi_category", spec["key"]]).to_csv(out_dir / f"{ds_name}_category_classes.csv",
                                                                 index=False)

        n_exemplars = {}
        for model in MODELS:
            ce = class_emb[model]
            row = {k: i for i, k in enumerate(ce[spec["npz_ids"]].tolist())}
            X_cmp = []
            for c in cats:
                idx = [row[k] for k in pools.loc[pools["cdi_category"] == c, spec["key"]]]
                n = ce["counts"][idx].sum()
                X_cmp.append(ce["sums"][idx].sum(axis=0) / n)
                n_exemplars[c] = int(n)
            tables = {ds_name: np.stack(X_cmp)}
            for ds in ("bv", "things"):
                raw = load_raw_table(src_dir / f"{ds}_{model}_exemplar_avg_raw_within_valid129.csv")
                missing = set(cats) - set(raw.index)
                assert not missing, f"{ds} {model} raw table lacks {missing}"
                tables[ds] = raw.loc[cats].to_numpy(dtype=np.float64)

            for ds, X in tables.items():
                stem = f"{ds}_{model}_exemplar_avg_{{}}_within_{set_name}.csv"
                save_table(out_dir / stem.format("raw"), cats, X)
                save_table(out_dir / stem.format("zscore"), cats, zscore_rows(X))

        meta = {
            "dataset": spec["description"],
            "category_set": set_name,
            "variant": variant,
            "n_categories": len(cats),
            f"n_{ds_name}_classes": int(pools[spec["key"]].nunique()),
            "n_exemplars_per_category": n_exemplars,
            "pooling": f"WordNet hyponym pooling, {spec['anchors']}",
            "exemplar_normalization": spec["exemplar"],
            "bv_things_source": "raw valid129 tables re-z-scored within this category set",
            "models": {m: str(class_emb[m]["embed_dir"]) for m in MODELS},
            "categories_alphabetical": cats,
        }
        (out_dir / f"{ds_name}_exemplar_embedding_run.json").write_text(json.dumps(meta, indent=2))
        print(f"{set_name}: {len(cats)} categories, {pools[spec['key']].nunique()} {ds_name} classes, "
              f"{sum(n_exemplars.values())} exemplars -> {out_dir}")

    return set_names

## Run (seconds)

In [6]:
set_names = build_category_sets(DATASET)
set_names

lvis_hyponym105: 105 categories, 285 lvis classes, 625761 exemplars -> /home/j7yang/babyview-projects/vss2026/object-detection/analysis/comparison-datasets/outputs/exemplar_sets/lvis_hyponym105
  exclusive ties -> larger pool: {'carnation': ['flower', 'plant'], 'sunflower': ['flower', 'plant']}


lvis_exclusive104: 104 categories, 285 lvis classes, 603785 exemplars -> /home/j7yang/babyview-projects/vss2026/object-detection/analysis/comparison-datasets/outputs/exemplar_sets/lvis_exclusive104


['lvis_hyponym105', 'lvis_exclusive104']

## Pools per category in the primary (exclusive) set

In [7]:
pools = pd.read_csv(EXEMPLAR_SETS / set_names[1] / f"{DATASET}_category_classes.csv")
label = "imagenet_synset" if DATASET == "imagenet" else "lvis_name"
pools.groupby("cdi_category")[label].agg(lambda s: ", ".join(map(str, s))).rename("pooled classes").to_frame().head(30)

,pooled classes
cdi_category,
alligator,alligator
apple,apple
ball,"baseball, basketball, bowling_ball, football_(American), ping-pong_ball, soccer_ball, softball, tennis_ball, volleyball"
balloon,"balloon, hot-air_balloon"
banana,banana
basket,"basket, clothes_hamper, hamper"
bathtub,bathtub
bed,"bed, bunk_bed"
belt,belt


## Sanity check: reproduces the manuscript
Re-z-scoring BV and THINGS on all 129 categories must give the manuscript's BV–THINGS RDM Spearman (CLIP 0.554, DINOv3 0.402).

In [8]:
# Dataset registry and raw-table loading (shared helpers; repeated in each notebook that needs them).
ALL_DATASETS = ("bv", "vedb", "things", "imagenet", "lvis")
LABEL = {"bv": "BV", "vedb": "VEDB", "things": "THINGS", "imagenet": "ImageNet", "lvis": "LVIS"}
COLOR = {"bv": "#90caf9", "vedb": "#0d47a1", "imagenet": "#eb6834", "lvis": "#1baf7a", "things": "#eda100"}
COMPARISON_SETS = {"imagenet": "imagenet_exclusive75", "lvis": "lvis_exclusive104"}


# zscore_rows: defined in the Stage 0 cell above (same formula).


def raw_path(ds: str, model: str) -> Path:
    if ds in COMPARISON_SETS:
        s = COMPARISON_SETS[ds]
        return EXEMPLAR_SETS / s / f"{ds}_{model}_exemplar_avg_raw_within_{s}.csv"
    if ds == "vedb":
        return VEDB_VALID129 / f"bv_{model}_exemplar_avg_raw_within_valid129.csv"
    return BV_THINGS_VALID129 / f"{ds}_{model}_exemplar_avg_raw_within_valid129.csv"


def load_raw(path: Path, cats: list[str]) -> np.ndarray:
    df = pd.read_csv(path)
    df["category"] = df["category"].astype(str).str.strip().str.lower()
    return df.set_index("category").loc[cats].filter(like="dim_").to_numpy(dtype=np.float64)


def shared_categories(datasets) -> list[str]:
    """Categories present in every given dataset's table, for both models (sorted)."""
    present = None
    for ds in datasets:
        for model in MODELS:
            cats = set(pd.read_csv(raw_path(ds, model), usecols=["category"])["category"].str.strip().str.lower())
            present = cats if present is None else present & cats
    return sorted(present)


def zscored_tables(datasets, cats, model) -> dict[str, np.ndarray]:
    """{dataset: category x dim matrix}, each z-scored within cats."""
    return {ds: zscore_rows(load_raw(raw_path(ds, model), cats)) for ds in datasets}

In [9]:
cats129 = shared_categories(["bv", "things"])
for model in MODELS:
    X = zscored_tables(["bv", "things"], cats129, model)
    print(model, len(cats129), "categories, BV-THINGS rho =",
          round(spearmanr(pdist(X["bv"], "cosine"), pdist(X["things"], "cosine"))[0], 3))

clip 129 categories, BV-THINGS rho = 0.554
dinov3 129 categories, BV-THINGS rho = 0.402


### Review checklist

- Does the exclusive rule assign each class to the right word (tie: LVIS sunflower → flower, not plant)?
- Are BV and THINGS re-z-scored on exactly the comparison dataset's categories?
- Does the sanity check print 0.554 and 0.402?